# TrashCam (Step-by-Step)

This notebook builds a simple **Python 3.11 desktop app** using:
- `tkinter` for the UI
- `opencv-python` for camera capture
- `transformers` models for classification
- `sqlite3` for local event logging and date-range reporting

The app uses a **two-model pipeline**:
1. Pretrained image model predicts what the item is.
2. Pretrained zero-shot text model decides whether it belongs in `recycle`, `compost`, or `landfill`.

The app has 2 tabs:
1. **Classifier**: Capture from webcam, classify item, assign disposal stream, and save to DB.
2. **Reports**: Pick start/end dates and get total counts by disposal stream.

> Note: This is still a simple prototype and should be treated as guidance, not a legal/local compliance system.


In [ ]:
# Step 1: Install dependencies (run once)
# If these are already installed, you can skip this cell.

%pip install -q opencv-python pillow numpy transformers torch torchvision


In [ ]:
# Step 2: Imports + global config

import sqlite3
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
from PIL import Image, ImageTk

import tkinter as tk
from tkinter import ttk, messagebox

from transformers import pipeline


BASE_DIR = Path.cwd()
SNAPSHOT_DIR = BASE_DIR / "snapshots"
DB_PATH = BASE_DIR / "trashcam.db"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# Target waste categories
WASTE_CANDIDATES = ["recycle", "compost", "landfill"]


In [ ]:
# Step 3: Load pretrained models (first run downloads weights)
# Model 1: image classifier -> predicts item name
# Model 2: zero-shot text classifier -> maps item name to recycle/compost/landfill

image_classifier = pipeline(
    task="image-classification",
    model="google/vit-base-patch16-224",
)

waste_router = pipeline(
    task="zero-shot-classification",
    model="facebook/bart-large-mnli",
)


def classify_item_and_disposal(frame_bgr: np.ndarray):
    rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    pil_img = Image.fromarray(rgb)

    item_preds = image_classifier(pil_img, top_k=5)
    top_item = item_preds[0]["label"]
    top_conf = float(item_preds[0]["score"])

    routing_text = (
        f"Item detected: {top_item}. "
        "Choose the best waste stream for this item in a typical household sorting context."
    )

    routing = waste_router(
        routing_text,
        candidate_labels=WASTE_CANDIDATES,
        hypothesis_template="This item should go to {}.",
    )

    disposal = routing["labels"][0]
    disposal_conf = float(routing["scores"][0])

    return top_item, top_conf, disposal, disposal_conf, item_preds


print("Pretrained models ready")


In [ ]:
# Step 4: SQLite database helpers


def init_db(db_path: Path = DB_PATH):
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            CREATE TABLE IF NOT EXISTS trash_events (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                event_time TEXT NOT NULL,
                item_label TEXT NOT NULL,
                disposal_type TEXT NOT NULL,
                confidence REAL NOT NULL,
                image_path TEXT
            )
            """
        )
        conn.commit()


def save_event(item_label: str, disposal_type: str, confidence: float, image_path: str | None = None, db_path: Path = DB_PATH):
    now = datetime.now().isoformat(timespec="seconds")
    with sqlite3.connect(db_path) as conn:
        conn.execute(
            """
            INSERT INTO trash_events (event_time, item_label, disposal_type, confidence, image_path)
            VALUES (?, ?, ?, ?, ?)
            """,
            (now, item_label, disposal_type, float(confidence), image_path),
        )
        conn.commit()


def report_counts(start_date: str, end_date: str, db_path: Path = DB_PATH):
    # expected input format: YYYY-MM-DD
    start_dt = datetime.fromisoformat(start_date + "T00:00:00")
    end_dt = datetime.fromisoformat(end_date + "T23:59:59")

    with sqlite3.connect(db_path) as conn:
        cur = conn.execute(
            """
            SELECT disposal_type, COUNT(*)
            FROM trash_events
            WHERE event_time BETWEEN ? AND ?
            GROUP BY disposal_type
            """,
            (start_dt.isoformat(timespec="seconds"), end_dt.isoformat(timespec="seconds")),
        )
        rows = cur.fetchall()

    counts = {"recycle": 0, "compost": 0, "landfill": 0}
    for disposal_type, total in rows:
        counts[disposal_type] = total
    return counts


init_db()
print("Database ready at", DB_PATH)


In [ ]:
# Step 5: Build the Tkinter desktop app


class TrashCamApp:
    def __init__(self, root: tk.Tk):
        self.root = root
        self.root.title("TrashCam Classifier")
        self.root.geometry("980x640")

        self.cap = cv2.VideoCapture(0)
        if not self.cap.isOpened():
            raise RuntimeError("Could not open camera. Check webcam permissions.")

        self.current_frame = None

        self.notebook = ttk.Notebook(root)
        self.notebook.pack(fill="both", expand=True)

        self.classifier_tab = ttk.Frame(self.notebook)
        self.report_tab = ttk.Frame(self.notebook)

        self.notebook.add(self.classifier_tab, text="Classifier")
        self.notebook.add(self.report_tab, text="Reports")

        self._build_classifier_tab()
        self._build_report_tab()
        self._update_preview()

        self.root.protocol("WM_DELETE_WINDOW", self.on_close)

    def _build_classifier_tab(self):
        left = ttk.Frame(self.classifier_tab, padding=12)
        left.pack(side="left", fill="both", expand=True)

        right = ttk.Frame(self.classifier_tab, padding=12)
        right.pack(side="right", fill="y")

        self.preview_label = ttk.Label(left)
        self.preview_label.pack(fill="both", expand=True)

        self.result_var = tk.StringVar(value="Waiting for capture...")
        ttk.Label(right, text="Last Classification", font=("Segoe UI", 12, "bold")).pack(anchor="w", pady=(0, 8))
        ttk.Label(right, textvariable=self.result_var, wraplength=290, justify="left").pack(anchor="w", pady=(0, 16))

        ttk.Button(right, text="Capture + Classify", command=self.capture_and_classify).pack(fill="x", pady=(0, 8))
        ttk.Button(right, text="Refresh Report Totals", command=self.run_report).pack(fill="x")

    def _build_report_tab(self):
        frame = ttk.Frame(self.report_tab, padding=12)
        frame.pack(fill="both", expand=True)

        ttk.Label(frame, text="Report Date Range", font=("Segoe UI", 12, "bold")).grid(row=0, column=0, columnspan=2, sticky="w", pady=(0, 10))

        ttk.Label(frame, text="Start date (YYYY-MM-DD)").grid(row=1, column=0, sticky="w")
        self.start_entry = ttk.Entry(frame, width=20)
        self.start_entry.grid(row=1, column=1, sticky="w", padx=(10, 0), pady=4)

        ttk.Label(frame, text="End date (YYYY-MM-DD)").grid(row=2, column=0, sticky="w")
        self.end_entry = ttk.Entry(frame, width=20)
        self.end_entry.grid(row=2, column=1, sticky="w", padx=(10, 0), pady=4)

        today = datetime.now().strftime("%Y-%m-%d")
        self.start_entry.insert(0, today)
        self.end_entry.insert(0, today)

        ttk.Button(frame, text="Run Report", command=self.run_report).grid(row=3, column=0, columnspan=2, sticky="w", pady=(10, 10))

        self.report_text = tk.Text(frame, width=60, height=10)
        self.report_text.grid(row=4, column=0, columnspan=2, sticky="nsew")

        frame.grid_columnconfigure(0, weight=1)
        frame.grid_rowconfigure(4, weight=1)

    def _update_preview(self):
        ok, frame = self.cap.read()
        if ok:
            self.current_frame = frame.copy()
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            pil_img = Image.fromarray(rgb)
            pil_img = pil_img.resize((720, 480))
            tk_img = ImageTk.PhotoImage(pil_img)
            self.preview_label.configure(image=tk_img)
            self.preview_label.image = tk_img

        self.root.after(30, self._update_preview)

    def capture_and_classify(self):
        if self.current_frame is None:
            messagebox.showwarning("No frame", "No camera frame available yet.")
            return

        frame = self.current_frame.copy()

        try:
            item_label, item_conf, disposal, disposal_conf, _ = classify_item_and_disposal(frame)
        except Exception as ex:
            messagebox.showerror("Classification error", str(ex))
            return

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        img_path = SNAPSHOT_DIR / f"capture_{timestamp}.jpg"
        cv2.imwrite(str(img_path), frame)

        # Save with item confidence as the core confidence metric
        save_event(item_label, disposal, item_conf, str(img_path))

        self.result_var.set(
            f"Item: {item_label}\n"
            f"Item confidence: {item_conf:.2f}\n"
            f"Disposal: {disposal}\n"
            f"Disposal confidence: {disposal_conf:.2f}\n"
            f"Saved: {img_path.name}"
        )

    def run_report(self):
        start_date = self.start_entry.get().strip()
        end_date = self.end_entry.get().strip()

        try:
            counts = report_counts(start_date, end_date)
        except ValueError:
            messagebox.showerror("Invalid date", "Use YYYY-MM-DD format for both dates.")
            return
        except Exception as ex:
            messagebox.showerror("Report error", str(ex))
            return

        lines = [
            f"Range: {start_date} to {end_date}",
            "",
            f"Recycle:  {counts['recycle']}",
            f"Compost:  {counts['compost']}",
            f"Landfill: {counts['landfill']}",
        ]

        self.report_text.delete("1.0", "end")
        self.report_text.insert("1.0", "\n".join(lines))

    def on_close(self):
        if self.cap is not None and self.cap.isOpened():
            self.cap.release()
        self.root.destroy()


In [ ]:
# Step 6: Run the app

root = tk.Tk()
app = TrashCamApp(root)
root.mainloop()


## How to Use

1. Run cells from top to bottom.
2. In the **Classifier** tab, click **Capture + Classify** when an item is in front of the camera.
3. Each capture is logged to `trashcam.db` with timestamp, predicted item, disposal type, confidence, and image path.
4. In the **Reports** tab, enter two dates (`YYYY-MM-DD`) and click **Run Report** to see totals.

## Notes

- First model load can be slow because pretrained weights are downloaded.
- This version does **not** hardcode specific item-to-bin mappings.
- If camera fails to open, check your OS webcam permissions for Python/Jupyter.
